In [ ]:
def r_search(query,k=3):
    query_embeddings = embedding_model.encode(query).astype('float32')
    query_embeddings = query_embeddings.reshape(1,-1)
    faiss.normalize_L2(query_embeddings)
    print(query_embeddings.shape)
    distance,index = index_faiss_db.search(query_embeddings,k=k)
    R_chunks = [chunks[i] for i in index[0]]
    R_str = ' '.join(R_chunks)
    return R_str

def g_text(r_search):
        import os
        import requests
    
        API_URL = "https://router.huggingface.co/v1/chat/completions"
    
        headers = {
            "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
        }
        def query(payload):
            response = requests.post(API_URL, headers=headers, json=payload)
            return response.json()
        prompt = f'''
                    You're an helpful assistant
                    Assigned Task for you : Structure my output => {r_search}
                    Note : 
                    1) Don't add extra contents just structure mentioned output.
                    2) If there is mistake in output correct or else keep the original output
                    with structured result.
            '''
        response = query({
            "messages": [
                {
                    "role": "user",
                    "content": f'{prompt}'  #JSONDecodeError
                }
            ],
            "model": "deepseek-ai/DeepSeek-R1:novita"
        })
    
        return response

user_prompt = 'Explain Machine Learning ?'
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)

r_response = r_search(user_prompt)
g_response = g_text(r_response)
print(g_response)